# ACC long-row shortener (Mad Hatters, 6 Oct)

**What it does:** takes our roads on the 42 rows where a single team holds the record and we are within 30 moves (longest road first; the cloud machine works the four 300-2,100-move rows, and 05698/06792/06018/08757 were already shortened) and shortens them with the exact window optimizer. The road is cut into pieces that run in parallel and are stitched back together, and every result is checked on the pinned official verifier.

**No model, no SAIR calls, no repo pull.** The kit is about 25 KB; the official verifier (23 MB, public) is cloned once per session.

**Steps:** 1) connect Drive, 2) unpack the kit, 3) run. **Keep this tab open.** Free Colab disconnects idle sessions.

Every improvement is saved at once to the `COLAB_LONGROWS` folder in the hub. If Colab disconnects, just **run all again**: it resumes from `state.jsonl` and skips rows it already finished.

Shorter roads go to `COLAB_LONGROWS/LINES_COLAB_LONGROWS.txt`, and UPG-H's check-in picks them up and checks them against the board for Jeremy to submit.

In [ ]:
#@title 1. Connect Drive
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
#@title 2. Unpack the kit from the hub
import zipfile, pathlib
HUB = pathlib.Path('/content/drive/MyDrive/Mad Hatters ACC Shared Hub')
zips = sorted(HUB.glob('ACC_LONGROWS_KIT_*.zip')) or sorted(pathlib.Path('/content/drive/MyDrive').rglob('ACC_LONGROWS_KIT_*.zip'))
assert zips, 'Put ACC_LONGROWS_KIT_*.zip in the Mad Hatters ACC Shared Hub folder'
KIT = pathlib.Path('/content/acc_kit')
with zipfile.ZipFile(zips[-1]) as z:
    z.extractall(KIT)
print('kit:', zips[-1].name, sorted(p.name for p in KIT.iterdir()))

In [ ]:
#@title 3. Run (about 11 hours; safe to stop and rerun, it resumes)
hours = 11 #@param {type:"number"}
rows = '' #@param {type:"string"}
import subprocess, sys, re
OUT = HUB / 'COLAB_LONGROWS'
DEFAULT = 'ac-09093,ac-08173,ac-08639,ac-05712,ac-09066,ac-00805,ac-05100,ac-03170,ac-09833,ac-09412,ac-07020,ac-07545,ac-00262,ac-08996,ac-04899,ac-09209,ac-09761,ac-00700,ac-06827,ac-08416,ac-02932,ac-00640,ac-07869,ac-00526,ac-05758,ac-07156,ac-00251,ac-00096,ac-01483,ac-10002,ac-02371,ac-04220,ac-01950,ac-01160,ac-03124,ac-06042,ac-08728,ac-03066,ac-09221,ac-09493,ac-04676,ac-06215'
todo = [r.strip() for r in (rows or DEFAULT).split(',') if r.strip()]
# a row counts as finished when the log moved on to another row after its 'done at' line
done, last = set(), None
log = OUT / 'log.txt'
for line in (log.read_text().splitlines() if log.exists() else []):
    m = re.search(r'(ac-\d{5}): (start|done at)', line)
    if m and m.group(2) == 'start':
        if last: done.add(last)
        last = None
    elif m:
        last = m.group(1)
todo = [r for r in todo if r not in done]
print(len(done), 'rows already finished;', len(todo), 'to go')
cmd = [sys.executable, str(KIT / 'run_longrows.py'), '--out', str(OUT), '--hours', str(hours), '--piece', '48', '--rows', ','.join(todo)]
p = subprocess.Popen(cmd, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True)
for line in p.stdout:
    print(line, end='', flush=True)
print('exit', p.wait())